# Subscription Renewal Prediction

**Classification Project 26 of 100** — Predict whether a subscriber will renew their subscription.

EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.


## 2. Project Overview

Predict whether a customer will renew their subscription based on engagement, plan type, tenure, payment, and service-use variables. Subscription businesses live and die by renewal rates — even small improvements in prediction accuracy can drive targeted retention efforts.

Binary classification with potential class imbalance depending on the product's natural renewal rate.


## 3. Learning Objectives

1. Binary classification for subscription/churn-style problems
2. Handling mixed numeric + categorical features
3. Feature engineering from subscription data (tenure, engagement ratios)
4. Dealing with potential class imbalance in renewal predictions
5. Comparing LazyPredict and FLAML on tabular data
6. Business-relevant metric selection (recall for at-risk subscribers)


## 4. Problem Statement

> Given subscriber profile and usage features, predict whether the subscriber will renew.

Binary classification. F1 and recall are important — missing an at-risk subscriber means losing them.


## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Product teams | Identify at-risk users early |
| Marketing | Target retention campaigns |
| Finance | Revenue forecasting accuracy |
| ML learners | Classic churn/renewal prediction pattern |


## 6. Dataset Overview

| Property | Value |
|---|---|
| Kaggle slug | priyamchoksi/subscription-renewal-prediction |
| Features | Customer demographics, subscription plan, usage, payment |
| Target | Renewal / non-renewal (binary) |
| Balance | Potentially imbalanced |


## 7. Dataset Source and License Notes

- Kaggle: priyamchoksi/subscription-renewal-prediction
- License: Check Kaggle page for specific license
- Subscription-style dataset with account-level attributes and binary renewal outcome.


## 8. Environment Setup


In [ ]:
import subprocess, sys, importlib
for pkg in ["lazypredict","flaml","kagglehub","xgboost","lightgbm"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")


## 9. Imports


In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay, classification_report)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); SEED=42
print("All imports loaded.")


## 10. Configuration / Constants


In [ ]:
KAGGLE_SLUG = "priyamchoksi/subscription-renewal-prediction"
TEST_SIZE = 0.15; VAL_SIZE = 0.15; SEED = 42; FLAML_BUDGET = 90
MAX_ROWS = 50000
print(f"Kaggle slug: {KAGGLE_SLUG}")


## 11. Dataset Download or Loading


In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Downloaded to: {path}")
except Exception as e:
    raise RuntimeError(f"Download failed: {e}")
csv_files = glob.glob(os.path.join(path,"**","*.csv"), recursive=True)
df_raw = pd.read_csv(csv_files[0])
print(f"Shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head()


## 12. Data Validation Checks


In [ ]:
print(f"Missing values:\n{df_raw.isnull().sum()}\n")
df = df_raw.copy()
target_candidates = [c for c in df.columns if any(h in c.lower() for h in ['renewal','renewed','target','churn'])]
if target_candidates:
    TARGET = target_candidates[0]
else:
    binary_cols = [c for c in df.columns if df[c].nunique() <= 5]
    TARGET = binary_cols[-1] if binary_cols else df.columns[-1]
print(f"Target column: {TARGET}")
id_cols = [c for c in df.columns if c.lower() in ['id','index','unnamed: 0','customerid','customer_id','user_id']]
if id_cols: df = df.drop(columns=id_cols); print(f"Dropped ID cols: {id_cols}")
dupes = df.duplicated().sum()
print(f"Duplicates: {dupes}")
if dupes > 0: df = df.drop_duplicates().reset_index(drop=True)
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f"Sampled to {MAX_ROWS}")
assert TARGET in df.columns
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")


## 13. Exploratory Data Analysis


In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
df[TARGET].value_counts().plot.bar(ax=ax, color=["steelblue","coral"])
ax.set_title(f"Target Distribution: {TARGET}")
plt.tight_layout(); plt.show()


In [ ]:
num_feats = df.select_dtypes(include=[np.number]).columns.tolist()
num_feats = [c for c in num_feats if c != TARGET]
if len(num_feats) >= 4:
    fig, axes = plt.subplots(2, 2, figsize=(12,8))
    for ax, col in zip(axes.flat, num_feats[:4]):
        for val in sorted(df[TARGET].unique()):
            ax.hist(df[df[TARGET]==val][col].dropna(), bins=25, alpha=0.4, label=f"{TARGET}={val}")
        ax.set_title(col); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()


In [ ]:
cat_feats = df.select_dtypes(include=['object','category']).columns.tolist()
if cat_feats:
    for col in cat_feats[:4]:
        print(f"{col}: {df[col].nunique()} unique")
if len(num_feats) >= 2:
    corr = df[num_feats + [TARGET]].corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(corr.iloc[:min(12,len(corr)),:min(12,len(corr))], annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Feature Correlation Heatmap')
    plt.tight_layout(); plt.show()
print(f"Numeric: {len(num_feats)}, Categorical: {len(cat_feats)}")


## 14. Target Analysis

Binary target — check balance. Subscription renewal rates are often skewed (most renew), so we may need class weighting or threshold tuning.


In [ ]:
print(df[TARGET].value_counts(normalize=True))
print(f"\nMajority baseline accuracy: {df[TARGET].value_counts(normalize=True).max():.1%}")


## 15. Train / Validation / Test Split

Stratified split to preserve class balance across train, validation, and test sets.


In [ ]:
X = df.drop(columns=[TARGET]); y = df[TARGET]
if y.dtype == object:
    le = LabelEncoder(); y = pd.Series(le.fit_transform(y), name=TARGET)
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED, stratify=y_tmp)
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")


## 16. Preprocessing Strategy

- **Numeric features**: Impute missing with median, then standardize
- **Categorical features**: Impute missing with mode, then one-hot encode
- Pipeline ensures no leakage from val/test into train


In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(("num", Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")


## 17. Feature Engineering

Create ratio and interaction features that capture subscriber engagement relative to tenure and plan cost.


In [ ]:
def engineer_features(d):
    d = d.copy()
    nc = d.select_dtypes(include=[np.number]).columns
    usage_cols = [c for c in nc if any(k in c.lower() for k in ['usage','login','session','activity','engage'])]
    tenure_cols = [c for c in nc if any(k in c.lower() for k in ['tenure','months','duration','age'])]
    if usage_cols and tenure_cols:
        d['USAGE_PER_TENURE'] = d[usage_cols[0]] / d[tenure_cols[0]].clip(lower=1)
    return d
X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()
transformers = []
if num_cols: transformers.append(("num", Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]), num_cols))
if cat_cols: transformers.append(("cat", Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
print(f"Total features after engineering: {X_train.shape[1]}")


## 18. Baseline Model

Start with a majority-class dummy, logistic regression, and random forest as sensible baselines.


In [ ]:
results = {}
for name, clf in [("Dummy", DummyClassifier(strategy="most_frequent", random_state=SEED)),
                  ("LogReg", LogisticRegression(max_iter=1000, random_state=SEED)),
                  ("RF", RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))]:
    pipe = Pipeline([("pre",preprocessor),("clf",clf)])
    pipe.fit(X_train, y_train); yp = pipe.predict(X_val)
    r = {"Accuracy": accuracy_score(y_val,yp), "F1": f1_score(y_val,yp,average='binary',zero_division=0),
         "Recall": recall_score(y_val,yp,average='binary',zero_division=0)}
    if hasattr(clf, "predict_proba"):
        try: r["ROC-AUC"] = roc_auc_score(y_val, pipe.predict_proba(X_val)[:,1])
        except: r["ROC-AUC"] = np.nan
    results[name] = r; print(f"{name}: {r}")


## 19. LazyPredict Benchmark

Quick scan of many classifiers to identify promising model families.


In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)


## 20. FLAML AutoML Run

Let FLAML automatically search for the best model and hyperparameters within the time budget.


In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task="classification", metric="f1",
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f"Best estimator: {automl.best_estimator}")
yp_fl = automl.predict(X_val_lp)
results["FLAML"] = {"Accuracy": accuracy_score(y_val, yp_fl),
                    "F1": f1_score(y_val, yp_fl, average='binary', zero_division=0)}
print(results["FLAML"])


## 21. Top 3 Model Selection

Based on LazyPredict and FLAML results, select three strong candidates for final evaluation.


In [ ]:
try:
    from lightgbm import LGBMClassifier; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBClassifier; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3["LightGBM"] = LGBMClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"] = ExtraTreesClassifier(n_estimators=400, random_state=SEED, n_jobs=-1)
if has_xgb:
    top3["XGBoost"] = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, n_jobs=-1, eval_metric="logloss")
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3["AdaBoost"] = AdaBoostClassifier(n_estimators=200, random_state=SEED)
top3["GradBoosting"] = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=SEED)
print(f"Top 3: {list(top3.keys())}")


## 22. Final Training and Evaluation of Top 3

Train on training set, evaluate on held-out test set with full classification metrics.


In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
labels = ["Not Renewed", "Renewed"]
final = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    ypr = mdl.predict_proba(X_te_t)[:,1]
    final[name] = {"Accuracy": accuracy_score(y_test,yp), "Precision": precision_score(y_test,yp,zero_division=0),
                   "Recall": recall_score(y_test,yp,zero_division=0), "F1": f1_score(y_test,yp,zero_division=0),
                   "ROC-AUC": roc_auc_score(y_test,ypr)}
    print(f"\n{name}:"); print(classification_report(y_test,yp,target_names=labels))
pd.DataFrame(final).T


In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(5*len(top3),4))
if len(top3)==1: axes=[axes]
for ax,(n,m) in zip(axes, top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, m.predict(X_te_t), ax=ax, cmap="Blues", display_labels=labels)
    ax.set_title(n)
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for n,m in top3.items():
    RocCurveDisplay.from_estimator(m, X_te_t, y_test, ax=ax, name=n)
ax.plot([0,1],[0,1],"k--",alpha=0.5); ax.set_title("ROC Curves")
plt.tight_layout(); plt.show()


## 23. Error Analysis

Examine false negatives (at-risk subscribers missed) and false positives (unnecessary retention spend).


In [ ]:
bm = list(top3.values())[0]; ypb = bm.predict(X_te_t)
fn = (y_test.values==1) & (ypb==0); fp = (y_test.values==0) & (ypb==1)
print(f"False Negatives (missed non-renewals): {fn.sum()}")
print(f"False Positives (unnecessary retention): {fp.sum()}")
print(f"Total misclassified: {(y_test.values!=ypb).sum()}/{len(y_test)} ({(y_test.values!=ypb).mean():.1%})")


## 24. Interpretation and Business Insight

Subscription renewal prediction helps businesses:
- **Proactive retention**: Offer incentives to at-risk subscribers before they churn
- **Resource allocation**: Focus customer success efforts on high-risk accounts
- **Revenue forecasting**: Better predict MRR (Monthly Recurring Revenue)

Key predictors typically include engagement frequency, tenure, payment history, and support ticket count.


## 25. Limitations

1. Static snapshot — no sequential behavior modeling
2. Missing engagement granularity (daily/weekly patterns)
3. No external factors (competitor pricing, market shifts)
4. Binary outcome — doesn't capture partial renewals or downgrades
5. Potential survivorship bias in the dataset


## 26. How to Improve This Project

1. Add time-series engagement features (rolling averages)
2. Include NPS/satisfaction survey scores
3. Survival analysis approach (time-to-churn)
4. Uplift modeling for retention campaign targeting
5. Deep learning on sequential usage logs


## 27. Production Considerations

- Schedule monthly batch scoring for renewal risk
- Set confidence thresholds for automated vs. manual review
- Integrate with CRM for automated retention workflows
- Monitor model drift as product features change
- A/B test retention interventions driven by model scores


## 28. Common Mistakes

1. Using accuracy when classes are imbalanced
2. Not stratifying train/test splits
3. Including post-renewal features (data leakage)
4. Ignoring seasonal renewal patterns
5. Not calibrating probabilities for business decisions


## 29. Mini Challenge / Exercises

1. Tune the decision threshold to maximize recall > 85%
2. Add class weights to handle imbalance — does F1 improve?
3. Remove the top 2 features — how much performance drops?
4. Compare SMOTE oversampling vs. class weights
5. Calculate the cost-benefit of each false negative vs. false positive


## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Binary — subscription renewal |
| Dataset | Subscription renewal data |
| Difficulty | Medium |
| Key insight | Engagement and tenure drive renewal |

Classic churn/renewal prediction pipeline with business-relevant evaluation, full workflow from EDA to deployment considerations.
